<a href="https://colab.research.google.com/github/fatmasenguler/Spanning-Tree_Thermostatics_of_Allostery/blob/main/3_cv_decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython networkx pandas matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.3 MB/s eta 0:00:00


In [ ]:
"""
3_cv_decomposition.py
=====================
Decomposes channel heat capacity: C_AB = C_AB,E + C_AB,T + C_AB,X

Reproduces: Table 2 (Delta_C_E, Delta_C_T, Delta_C_X columns).

Dependencies: numpy (standalone, Colab-compatible)

Author: Fatma Ciftci & Burak Erman
"""

import numpy as np
from collections import defaultdict
import os

CUTOFF = 7.8
KT = 1.0
MAX_LEN = 9

CHANNELS = [
    (6, 11,   "Ch1: P-loop (6-11)"),
    (55, 60,  "Ch2: pre-Switch II (55-60)"),
    (110, 117, "Ch3: GBS (110-117)"),
    (141, 146, "Ch4: SAK motif (141-146)"),
    (19, 142,  "Ch5: Inter-lobe linker (19-142)"),
]


def parse_pdb_ca(filename):
    coords = {}
    with open(filename) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                atom_name = line[12:16].strip()
                if atom_name == "CA":
                    resid = int(line[22:26].strip())
                    x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                    if resid not in coords:
                        coords[resid] = np.array([x, y, z])
    return coords


def build_contact_graph(coords, cutoff, kT):
    residues = sorted(coords.keys())
    adj = defaultdict(dict)
    dist = {}
    for i, ri in enumerate(residues):
        for j in range(i + 1, len(residues)):
            rj = residues[j]
            d = np.linalg.norm(coords[ri] - coords[rj])
            if d <= cutoff:
                w = np.exp(-d / kT)
                adj[ri][rj] = w
                adj[rj][ri] = w
                dist[(ri, rj)] = d
                dist[(rj, ri)] = d
    nodes = sorted(set(adj.keys()))
    return adj, nodes, dist


def build_laplacian(adj, nodes):
    n = len(nodes)
    node_idx = {r: i for i, r in enumerate(nodes)}
    L = np.zeros((n, n))
    for ri in nodes:
        for rj, w in adj[ri].items():
            if rj in node_idx:
                i, j = node_idx[ri], node_idx[rj]
                L[i, j] -= w
                L[i, i] += w
    Lplus = np.linalg.pinv(L)
    return L, Lplus, node_idx


def enumerate_paths(adj, source, target, max_len):
    if source not in adj or target not in adj:
        return []
    all_paths = []
    stack = [(source, [source])]
    while stack:
        node, path = stack.pop()
        if len(path) > max_len:
            continue
        if node == target and len(path) > 1:
            all_paths.append(path[:])
            continue
        for neighbor in adj[node]:
            if neighbor not in path and len(path) < max_len:
                stack.append((neighbor, path + [neighbor]))
    return all_paths


def path_weight_decomposed(path, adj, Lplus, node_idx, dist_dict, kT):
    edges = [(path[k], path[k + 1]) for k in range(len(path) - 1)]
    m = len(edges)
    if m == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    E_phys = sum(dist_dict[(ei, ej)] for ei, ej in edges)
    B = np.exp(-E_phys / kT)

    Y = np.zeros((m, m))
    for a, (ia, ja) in enumerate(edges):
        for b, (ib, jb) in enumerate(edges):
            Y[a, b] = (Lplus[node_idx[ia], node_idx[ib]] +
                       Lplus[node_idx[ja], node_idx[jb]] -
                       Lplus[node_idx[ia], node_idx[jb]] -
                       Lplus[node_idx[ja], node_idx[ib]])

    tau = np.linalg.det(Y)

    K_pi = np.zeros((m, m))
    for a, (ia, ja) in enumerate(edges):
        w_a = adj[ia][ja]
        for b in range(m):
            K_pi[a, b] = w_a * Y[a, b]
    W = np.linalg.det(K_pi)
    E_topo = -kT * np.log(tau) if tau > 0 else np.inf

    return W, E_phys, E_topo, B, tau


def compute_cv_decomposition(paths, adj, Lplus, node_idx, dist_dict, kT):
    n = len(paths)
    if n == 0:
        return None

    W_arr, EE_arr, ET_arr = np.zeros(n), np.zeros(n), np.zeros(n)
    valid = np.ones(n, dtype=bool)

    for i, path in enumerate(paths):
        W, E_phys, E_topo, B, tau = path_weight_decomposed(path, adj, Lplus, node_idx, dist_dict, kT)
        W_arr[i] = max(W, 0.0)
        EE_arr[i] = E_phys
        if np.isinf(E_topo) or tau <= 0:
            valid[i] = False
        else:
            ET_arr[i] = E_topo

    mask = valid & (W_arr > 0)
    if mask.sum() == 0:
        return None

    W_v, EE_v, ET_v = W_arr[mask], EE_arr[mask], ET_arr[mask]
    Z_AB = W_v.sum()
    P = W_v / Z_AB
    E_total = EE_v + ET_v

    mean_EE, mean_ET, mean_E = np.sum(P*EE_v), np.sum(P*ET_v), np.sum(P*E_total)
    var_EE = np.sum(P * (EE_v - mean_EE)**2)
    var_ET = np.sum(P * (ET_v - mean_ET)**2)
    cov = np.sum(P * (EE_v - mean_EE) * (ET_v - mean_ET))
    kT2 = kT * kT

    return {
        'Z_AB': Z_AB, 'n_paths': int(mask.sum()), 'n_total': n,
        'mean_EE': mean_EE, 'mean_ET': mean_ET, 'mean_E': mean_E,
        'C_E': var_EE/kT2, 'C_T': var_ET/kT2, 'C_X': 2.0*cov/kT2,
        'C_total': np.sum(P*(E_total-mean_E)**2)/kT2,
    }


def main():
    try:
        from google.colab import files as colab_files
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    pdb_files = {"WT": "6GOD.pdb", "G12D": "6GOF.pdb"}
    for label, fname in pdb_files.items():
        if not os.path.exists(fname):
            if IN_COLAB:
                print(f"Upload {fname}:"); colab_files.upload()
            else:
                raise FileNotFoundError(f"{fname} not found.")

    all_results = {}
    for label, fname in pdb_files.items():
        print(f"\n{'='*70}\n  {label} ({fname})\n{'='*70}")
        coords = parse_pdb_ca(fname)
        adj, nodes, dist_dict = build_contact_graph(coords, CUTOFF, KT)
        L, Lplus, node_idx = build_laplacian(adj, nodes)
        all_results[label] = {}

        for source, target, ch_label in CHANNELS:
            print(f"\n  --- {ch_label} ---")
            if source not in node_idx or target not in node_idx:
                print(f"  WARNING: endpoints not in graph!"); continue
            paths = enumerate_paths(adj, source, target, MAX_LEN)
            print(f"  Paths: {len(paths)}")
            if not paths: continue
            res = compute_cv_decomposition(paths, adj, Lplus, node_idx, dist_dict, KT)
            if res is None: continue
            all_results[label][ch_label] = res
            print(f"  C_E={res['C_E']:.4f}  C_T={res['C_T']:.4f}  C_X={res['C_X']:.4f}  C_total={res['C_total']:.4f}")

    # Difference table
    print(f"\n{'='*80}\n  DIFFERENCES (G12D - WT)\n{'='*80}")
    for source, target, ch_label in CHANNELS:
        rw = all_results.get('WT', {}).get(ch_label)
        rm = all_results.get('G12D', {}).get(ch_label)
        if rw and rm:
            print(f"  {ch_label:<30}  DC_E={rm['C_E']-rw['C_E']:+.4f}  "
                  f"DC_T={rm['C_T']-rw['C_T']:+.4f}  DC_X={rm['C_X']-rw['C_X']:+.4f}  "
                  f"DC={rm['C_total']-rw['C_total']:+.4f}")

    outfile = "cv_decomposition_results.txt"
    with open(outfile, 'w') as f:
        f.write("HEAT CAPACITY DECOMPOSITION: C_AB = C_AB,E + C_AB,T + C_AB,X\n")
        f.write(f"Cutoff={CUTOFF} A, kT={KT}, L_max={MAX_LEN}\n\n")
        for source, target, ch_label in CHANNELS:
            for ls in ['WT', 'G12D']:
                r = all_results.get(ls, {}).get(ch_label)
                if r:
                    f.write(f"{ch_label} {ls}: C_E={r['C_E']:.6f} C_T={r['C_T']:.6f} "
                            f"C_X={r['C_X']:.6f} C_total={r['C_total']:.6f}\n")
    print(f"\nSaved: {outfile}")


if __name__ == "__main__":
    main()


Upload 6GOD.pdb:


Saving 6GOD.pdb to 6GOD.pdb
Upload 6GOF.pdb:


Saving 6GOF.pdb to 6GOF.pdb

  WT (6GOD.pdb)

  --- Ch1: P-loop (6-11) ---
  Paths: 946282
  C_E=49.0282  C_T=17.4294  C_X=-54.8310  C_total=11.6265

  --- Ch2: pre-Switch II (55-60) ---
  Paths: 638606
  C_E=41.9096  C_T=16.9735  C_X=-49.5931  C_total=9.2901

  --- Ch3: GBS (110-117) ---
  Paths: 824774
  C_E=23.3624  C_T=6.9678  C_X=-21.4652  C_total=8.8650

  --- Ch4: SAK motif (141-146) ---
  Paths: 934827
  C_E=47.8882  C_T=16.9440  C_X=-53.4909  C_total=11.3412

  --- Ch5: Inter-lobe linker (19-142) ---
  Paths: 864439
  C_E=46.3465  C_T=17.6777  C_X=-54.2002  C_total=9.8241

  G12D (6GOF.pdb)

  --- Ch1: P-loop (6-11) ---
  Paths: 968240
  C_E=48.5671  C_T=17.2957  C_X=-54.3799  C_total=11.4830

  --- Ch2: pre-Switch II (55-60) ---
  Paths: 642205
  C_E=41.8587  C_T=16.9619  C_X=-49.5717  C_total=9.2489

  --- Ch3: GBS (110-117) ---
  Paths: 858454
  C_E=23.6236  C_T=6.9666  C_X=-21.5966  C_total=8.9937

  --- Ch4: SAK motif (141-146) ---
  Paths: 964794
  C_E=47.7094  C_T=16.89